# Lab: Hồi quy Logistic (Logistic Regression)

## 1. Vì sao không dùng Linear Regression cho phân loại?

Linear Regression cho output là một số thực bất kỳ — có thể âm, có thể >1. Trong phân loại nhị phân, ta muốn output ∈ [0, 1] hiểu là *xác suất* thuộc lớp 1.

Giải pháp: **đưa $w^T x + b$ qua hàm sigmoid** để ép về [0, 1]:
$$
\hat{p}(x) = \sigma(w^T x + b) = \frac{1}{1 + e^{-(w^T x + b)}}
$$

Đây là **Logistic Regression** — nghe tên thì giống regression nhưng thực chất là **classifier**.

## 2. Hàm Sigmoid

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Tính chất:
- $\sigma(0) = 0.5$.
- $\sigma(z) \to 1$ khi $z \to +\infty$, $\sigma(z) \to 0$ khi $z \to -\infty$.
- Đạo hàm rất đẹp: $\sigma'(z) = \sigma(z)(1 - \sigma(z))$.

Sau khi có $\hat{p}$, dự đoán nhãn theo ngưỡng (mặc định 0.5):
$$\hat{y} = \begin{cases}1 & \hat{p} \ge 0.5 \\ 0 & \hat{p} < 0.5\end{cases}$$

## 3. Hàm mất mát: Binary Cross-Entropy

Vì sao **không** dùng MSE? MSE + sigmoid → loss landscape có nhiều local minima → khó train. Hơn nữa, gradient gần 0 ở vùng $\hat{p}$ gần 0 hoặc 1 (sigmoid bão hoà).

Logistic Regression dùng **Binary Cross-Entropy (BCE)**:
$$
L = -\frac{1}{N}\sum_{i=1}^{N}\Big[y_i \log \hat{p}_i + (1 - y_i) \log(1 - \hat{p}_i)\Big]
$$

Đọc bằng lời:
- Nếu $y_i = 1$: muốn $\hat{p}_i$ gần 1 → $-\log \hat{p}_i$ nhỏ.
- Nếu $y_i = 0$: muốn $\hat{p}_i$ gần 0 → $-\log(1 - \hat{p}_i)$ nhỏ.

BCE convex theo $w$ → Gradient Descent đảm bảo về cực tiểu toàn cục.

## 4. Một công thức gradient cực đẹp

Khi tổ hợp Sigmoid + BCE, gradient có dạng đơn giản:
$$\frac{\partial L}{\partial w_j} = \frac{1}{N}\sum_{i=1}^{N}(\hat{p}_i - y_i) \cdot x_{i,j}$$

Nghĩa là: gradient = (sai số) × (input). Đây cũng chính là gradient của Linear Regression với MSE — chỉ khác $\hat{p}$ là sigmoid thay vì linear. Đây là một *vẻ đẹp toán học* sâu sắc đứng đằng sau cả deep learning.

## 5. Mở rộng cho Multiclass: Softmax Regression

Với $C$ lớp, ta dùng:
$$\hat{p}_c(x) = \frac{e^{w_c^T x + b_c}}{\sum_{k=1}^{C} e^{w_k^T x + b_k}}$$

Đây là **softmax** — hàm xác suất nhiều chiều. Loss tương ứng là **categorical cross-entropy**:
$$L = -\frac{1}{N}\sum_{i=1}^{N}\sum_{c=1}^{C} y_{i,c} \log \hat{p}_{i,c}$$

Trong sklearn, `LogisticRegression(multi_class='multinomial')` chính là Softmax Regression.

# THỰC HÀNH 1: Logistic Regression từ scratch trên data 2D

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, load_breast_cancer, load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_curve, auc)

np.random.seed(42)

# Sinh dữ liệu 2D có thể phân loại tuyến tính được
X, y = make_classification(n_samples=200, n_features=2, n_redundant=0,
                            n_informative=2, n_clusters_per_class=1,
                            random_state=42)

plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolor='k', s=30)
plt.title('Dữ liệu 2D, 2 lớp'); plt.grid(alpha=0.3); plt.show()

In [ ]:
# Cài Logistic Regression bằng Gradient Descent thủ công
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

class MyLogReg:
    def __init__(self, lr=0.1, n_iter=2000):
        self.lr = lr
        self.n_iter = n_iter

    def fit(self, X, y):
        N, d = X.shape
        self.w = np.zeros(d)
        self.b = 0.0
        self.loss_hist = []
        for _ in range(self.n_iter):
            z = X @ self.w + self.b
            p = sigmoid(z)
            # BCE loss (cộng epsilon tránh log 0)
            loss = -np.mean(y * np.log(p + 1e-9) + (1 - y) * np.log(1 - p + 1e-9))
            self.loss_hist.append(loss)
            # gradient
            dw = (X.T @ (p - y)) / N
            db = (p - y).mean()
            self.w -= self.lr * dw
            self.b -= self.lr * db
        return self

    def predict_proba(self, X):
        return sigmoid(X @ self.w + self.b)

    def predict(self, X, thresh=0.5):
        return (self.predict_proba(X) >= thresh).astype(int)

mine = MyLogReg(lr=0.1, n_iter=2000).fit(X, y)
print(f'My LR     accuracy: {(mine.predict(X) == y).mean()*100:.2f}%')
print(f'w = {mine.w}, b = {mine.b:.4f}')

skl = LogisticRegression(C=1e10).fit(X, y)   # C lớn = ít regularization, khớp model thuần
print(f'sklearn   accuracy: {skl.score(X, y)*100:.2f}%')
print(f'w = {skl.coef_[0]}, b = {skl.intercept_[0]:.4f}')

In [ ]:
# Vẽ decision boundary
xx, yy = np.meshgrid(np.linspace(X[:, 0].min()-1, X[:, 0].max()+1, 200),
                      np.linspace(X[:, 1].min()-1, X[:, 1].max()+1, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
proba = mine.predict_proba(grid).reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
cs = axes[0].contourf(xx, yy, proba, levels=20, cmap='coolwarm', alpha=0.7)
axes[0].contour(xx, yy, proba, levels=[0.5], colors='black', linestyles='--')
axes[0].scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolor='white', s=30)
plt.colorbar(cs, ax=axes[0], label='P(y=1)')
axes[0].set_title('Xác suất + boundary 0.5'); axes[0].grid(alpha=0.3)

axes[1].plot(mine.loss_hist); axes[1].set_xlabel('Iteration'); axes[1].set_ylabel('BCE loss')
axes[1].set_title('Loss giảm dần qua GD'); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

# THỰC HÀNH 2: Phân loại ung thư trên Breast Cancer

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

# QUAN TRỌNG: scale feature trước Logistic Regression khi có regularization
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_s, y_train)
y_pred  = model.predict(X_test_s)
y_proba = model.predict_proba(X_test_s)[:, 1]

print(classification_report(y_test, y_pred, target_names=data.target_names))

In [ ]:
# ROC + AUC
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc_val = auc(fpr, tpr)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC
axes[0].plot(fpr, tpr, linewidth=2, label=f'AUC = {auc_val:.3f}')
axes[0].plot([0, 1], [0, 1], 'k--', label='Đoán mò')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curve'); axes[0].legend(); axes[0].grid(alpha=0.3)

# Top 10 feature importance
imp = pd.Series(np.abs(model.coef_[0]), index=data.feature_names).nlargest(10)
imp.plot.barh(ax=axes[1])
axes[1].set_xlabel('|hệ số|'); axes[1].set_title('Top 10 feature quan trọng nhất')
plt.tight_layout(); plt.show()

## 6. Threshold tuning

Mặc định predict ở ngưỡng 0.5. Trong y học, có khi muốn giảm bỏ sót (recall cao hơn) — chấp nhận nhiều báo động giả. Đổi ngưỡng để cân bằng precision/recall theo nhu cầu.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.arange(0.1, 0.95, 0.05)
precs, recs, f1s = [], [], []
for t in thresholds:
    pred = (y_proba >= t).astype(int)
    precs.append(precision_score(y_test, pred, zero_division=0))
    recs.append(recall_score(y_test, pred))
    f1s.append(f1_score(y_test, pred))

plt.figure(figsize=(8, 4.5))
plt.plot(thresholds, precs, 'o-', label='Precision')
plt.plot(thresholds, recs, 's-', label='Recall')
plt.plot(thresholds, f1s, '^-', label='F1')
best_t = thresholds[np.argmax(f1s)]
plt.axvline(best_t, color='red', linestyle='--', label=f'Best F1 ở {best_t:.2f}')
plt.xlabel('Ngưỡng quyết định'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Precision / Recall / F1 thay đổi theo ngưỡng')
plt.show()

# THỰC HÀNH 3: Multiclass Logistic (Softmax) trên Iris

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# multi_class='multinomial' = softmax thực sự (chứ không phải one-vs-rest)
model = LogisticRegression(multi_class='multinomial', max_iter=1000)
model.fit(X_train_s, y_train)
y_pred = model.predict(X_test_s)

print(f'Test accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%')
print(classification_report(y_test, y_pred, target_names=iris.target_names))

## Tổng kết

1. Logistic Regression = Linear Regression + Sigmoid + BCE → cho xác suất phân loại nhị phân.
2. **Phải scale feature** khi có regularization (mặc định sklearn dùng L2).
3. **Đừng dùng MSE** với sigmoid — non-convex và gradient bão hoà. Luôn dùng BCE.
4. Multi-class: dùng **softmax** + cross-entropy.
5. **Threshold** mặc định 0.5 — có thể tune theo bài toán cụ thể (ưu tiên precision hay recall).
6. Logistic Regression đơn giản nhưng cực kỳ mạnh — luôn nên thử trước khi dùng model phức tạp.

# BÀI TẬP VỀ NHÀ

## Bài 1: Polynomial features cho Logistic
Trên `make_moons` (dữ liệu cong không tách tuyến tính được):
1. Train Logistic Regression thường — bao nhiêu accuracy?
2. Thêm `PolynomialFeatures(degree=3)` trước khi fit. Bao nhiêu accuracy?
3. Vẽ decision boundary cho cả hai. Quan sát: Logistic + polynomial cũng học được boundary cong.

## Bài 2: Regularization C
Trên Breast Cancer, sweep `C ∈ {0.001, 0.01, 0.1, 1, 10, 100, 1000}`. (Lưu ý: trong sklearn, $C = 1/\alpha$ — C nhỏ = regularization mạnh.)

1. Vẽ test accuracy theo C (log scale).
2. Vẽ ||w||² theo C.
3. Quan sát: C nhỏ → coef nhỏ, C lớn → coef lớn → có thể overfit.

## Bài 3: L1 vs L2
Train Logistic Regression với `penalty='l2'` và `penalty='l1'` (cần `solver='liblinear'` cho L1). Đếm số coef khác 0 trong mỗi trường hợp. L1 có "chọn feature" giống Lasso không?

## Bài 4: Class imbalance
Sinh dữ liệu 95/5 mất cân bằng. Train Logistic. Báo cáo accuracy, F1 cho lớp ít.

Sau đó dùng `class_weight='balanced'`. So sánh F1 lớp ít trước vs sau.

## Bài 5: So sánh với Linear SVM và KNN
Trên Breast Cancer (đã scale), train 3 model:
1. `LogisticRegression()`
2. `LinearSVC()` hoặc `SVC(kernel='linear')`
3. `KNeighborsClassifier(n_neighbors=7)`

So sánh accuracy + F1. Cái nào ổn nhất? Vì sao Logistic và Linear SVM thường rất gần nhau? *Cả hai đều tìm hyperplane tuyến tính, chỉ khác ở loss function.*